# Run sweep seeds on Kaggle (one seed per GPU)

**One-time setup**
1. *Datasets → New Dataset*: upload `influence-pruning.zip` (name it e.g. `influence-pruning-code`). It's fine if Kaggle unpacks it.
2. *Code → New Notebook*, then *File → Import Notebook* and choose this file.
3. Right panel: *Add Input* → your `influence-pruning-code` dataset.
4. *Settings*: **Accelerator = GPU T4 x2** and **Internet = On**. Internet needs a phone-verified account; it's used to download CIFAR.
   Use T4 x2 rather than P100: recent PyTorch builds may not support the P100, and two GPUs let both seeds run at the same time.

**Each session**
5. *Save Version → Save & Run All (Commit)*. The notebook runs in the background, so you can close the browser.
   The sweep stops starting new jobs after `MAX_HOURS`, so the session finishes under Kaggle's 12 h limit and the output is saved.
6. If the sweep isn't finished (the last cell prints the remaining jobs): *Add Input → Notebook Output* of **this notebook's latest version**, then *Save & Run All* again.
   The notebook copies the previous `runs/` and resumes. Finished runs are skipped.

**When done**: download the output and copy `runs/<name>/results/*.json` into the PC's `runs/<name>/results/`.
Keep the whole downloaded `runs/` folder: later sweeps can reuse each seed's scoring run.

In [ ]:
SWEEP     = "sweeps/tier5_finetune.yaml"
SEEDS     = ["0", "1", "2"]   # spread round-robin over the GPUs (T4 x2: seeds 0+2 on GPU 0, seed 1 on GPU 1)
MAX_HOURS = 10.0             # no new job starts after this; leaves ~2 h margin before the 12 h limit

In [ ]:
import glob, os, shutil, subprocess, sys, time
REPO, DATA, OUT = "/tmp/influence-pruning", "/tmp/data", "/kaggle/working/runs"
shutil.rmtree(REPO, ignore_errors=True)
zips = glob.glob("/kaggle/input/**/influence-pruning.zip", recursive=True)
if zips:
    subprocess.run(["unzip", "-oq", zips[0], "-d", "/tmp"], check=True)
else:  # Kaggle unpacked the zip
    found = [os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/dpif/sweep.py", recursive=True)]
    assert found, "code not found: add the dataset containing influence-pruning.zip as input"
    shutil.copytree(found[0], REPO)
os.chdir(REPO)

# restore outputs of earlier sessions (attached as 'Notebook Output' inputs)
os.makedirs(OUT, exist_ok=True)
for d in glob.glob("/kaggle/input/**/runs", recursive=True):
    if os.path.isdir(d) and glob.glob(d + "/*/results"):
        print("restoring", d)
        subprocess.run(f'cp -rn "{d}/." "{OUT}/"', shell=True, check=True)
if os.path.lexists("runs"):
    shutil.rmtree("runs") if os.path.isdir("runs") and not os.path.islink("runs") else os.remove("runs")
os.symlink(OUT, "runs")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabulate"], check=True)
import torch
print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# quick checks + one-time downloads, done ONCE before the parallel processes start
# (two processes downloading the same file at the same time can corrupt it)
import yaml
sw = yaml.safe_load(open(SWEEP)); cfg = yaml.safe_load(open(sw["config"]))
NAME = sw.get("name") or cfg["name"]
print(subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True).stdout.splitlines()[-1])
subprocess.run([sys.executable, "-c", f"from dpif.data import _load_raw; _load_raw('{cfg['data']['dataset']}', '{DATA}')"], check=True)
if cfg["model"].get("pretrained"):
    subprocess.run([sys.executable, "-c", f"import torchvision; torchvision.models.{cfg['model']['arch']}(weights='DEFAULT')"], check=True)
    print("pretrained weights cached")
if any(m.get("emb") == "dinov2_s14" for m in sw["methods"]):
    subprocess.run([sys.executable, "-m", "dpif.embed", "--config", sw["config"], "--seed", SEEDS[0], "--kind", "dinov2_s14",
                    "--run-dir", f"runs/{NAME}", "--device", "cuda", "--set", f"data.root={DATA}"], check=True)
    print("DINOv2 embedding ready:", os.path.exists(f"runs/{NAME}/emb_dinov2_s14.npy"))

In [ ]:
# long-running cell
ngpu = torch.cuda.device_count()
groups = [g for g in (SEEDS[i::max(ngpu, 1)] for i in range(max(ngpu, 1))) if g]  # round-robin seeds over GPUs
procs = []
for i, g in enumerate(groups):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i % max(ngpu, 1)))
    log = open(f"{OUT}/kaggle_seed{'-'.join(g)}.log", "a")
    cmd = [sys.executable, "-m", "dpif.sweep", SWEEP, "--seeds", ",".join(g), "--device", "cuda",
           "--max-hours", str(MAX_HOURS), "--set", f"data.root={DATA}"]
    procs.append(subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env))
    print("started", " ".join(cmd), "on GPU", env["CUDA_VISIBLE_DEVICES"])
while any(p.poll() is None for p in procs):
    time.sleep(60)
    for g in groups:
        lines = [l for l in open(f"{OUT}/kaggle_seed{'-'.join(g)}.log") if l.startswith("[")]
        print(time.strftime("%H:%M"), "seeds", g, "|", lines[-1].strip() if lines else "starting", flush=True)
print("exit codes:", [p.returncode for p in procs])

In [ ]:
# what is done / what remains
for g in groups:
    print(subprocess.run([sys.executable, "-m", "dpif.sweep", SWEEP, "--seeds", ",".join(g), "--dry-run",
                          "--device", "cuda", "--set", f"data.root={DATA}"], capture_output=True, text=True).stdout)
print(subprocess.run([sys.executable, "-m", "dpif.aggregate", f"runs/{NAME}"], capture_output=True, text=True).stdout)